In [1]:
%load_ext autoreload
%autoreload 2
import os,shutil
import numpy as np
import pandas as pd
import pickle as pkl

from inDecay import PATH
os.chdir(PATH.main_dir)

from qrguide import analysis_fn

In [2]:
from inDecay import my_utils

In [3]:
BOB = 'ST_June_2017_BOB_LV7A_DPI7'
ETG = 'ST_June_2017_E14TG2A_LV7A_DPI7'
CHO= 'ST_June_2017_CHO_LV7A_DPI7'
HAP1 = 'ST_June_2017_HAP1_LV7A_DPI7'
K562= 'ST_June_2017_K562_LV7A_DPI7'

celltype_rename = { BOB:'iPSC', 
                    ETG:'mESC',
                    CHO: 'CHO',
                    HAP1: 'HAP1',
                    K562: 'K562',}

In [4]:
def read_pkl(path):
    with open(path, 'rb') as f:
        Y = pkl.load(f)
    f.close()
    return Y

def evalute_fn(Y_true_path, Y_pred_path):
    Y_pred =read_pkl(Y_pred_path)
    Y = read_pkl(Y_true_path)

    eval_json = analysis_fn.assessment_recipe_forecast(Y, Y_pred)
    eval_json.update(analysis_fn.assessment_recipe_IDL_forecast(Y, Y_pred))

    eval_df = pd.json_normalize(eval_json)
    return eval_df


# find the ckpt

In [5]:
pred_pkl_path_dict = {}
true_pkl_path_dict = {}


for feat_i in ['C100','C200','C500']:

    pred_pkl_path_dict[feat_i] = {}
    true_pkl_path_dict[feat_i] = {}

    for exp in [BOB, ETG, HAP1, CHO, K562]:
        
        celltype = celltype_rename[exp]

        # point to lightning saving path    
        log_path = os.path.join(
            f"pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_{feat_i}",
            exp,
            "lightning_logs"
        )

        if not os.path.exists(log_path):
            continue

        versions = [f for f in os.listdir(log_path) if f.startswith('version')]

        # remove empty versions
        if len(versions) > 1:
            for v in versions:
                 if 'checkpoints' not in os.listdir(os.path.join(log_path, v)):
                     shutil.rmtree( os.path.join(log_path, v))
                     
        try:
            pkl_path = my_utils.find_ckpt(log_path).replace(".ckpt", "TestPred.pkl")
            pred_pkl_path_dict[feat_i][celltype] = pkl_path

            Ytrue_path = log_path.replace("lightning_logs","ForeCast_TestY.pkl")
            true_pkl_path_dict[feat_i][celltype] = Ytrue_path

        except FileNotFoundError:
            # skip some feat_i or celltype if no valid ckpt found
            continue    

    print(f"for {feat_i}, checkpoints of {len(pred_pkl_path_dict[feat_i])} cells were found")

for C100, checkpoints of 5 cells were found
for C200, checkpoints of 5 cells were found
for C500, checkpoints of 5 cells were found


In [6]:
for feat_i in ['C100','C200','C500']:
    print(pred_pkl_path_dict[feat_i])

{'iPSC': 'pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C100/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=89-val_cre=2.96463704TestPred.pkl', 'mESC': 'pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C100/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_1/checkpoints/epoch=95-val_cre=3.02341413TestPred.pkl', 'HAP1': 'pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C100/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_10/checkpoints/epoch=99-val_cre=3.25392866TestPred.pkl', 'CHO': 'pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C100/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_1/checkpoints/epoch=97-val_cre=3.27636862TestPred.pkl', 'K562': 'pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C100/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_1/checkpoints/epoch=88-val_cre=3.34550405TestPred.pkl'}
{'iPSC': 'pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C200/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=98

# evaluation

In [17]:
perform

[   KL divergence  Top5 events recall  Top10 events recall  \
 0       2.014625            3.120035             5.831421   
 
    R2 of Frameshift ratio  Coll_I_Top5  Coll_I_Top10   KLD_IDL  Top5_IDL  \
 0                0.850139     3.225066      5.964695  0.237909  3.467785   
 
    Top10_IDL  W1-distance_IDL  delratio_r2  Kendall_tau_IDL celltype  \
 0   7.029126         0.071106     0.790562         0.674679     iPSC   
 
   feat_range  
 0       C100  ]

In [7]:
perform = []
for feat_range, pred_pkls in pred_pkl_path_dict.items():

    true_pkls = true_pkl_path_dict[feat_range]
    try:
        for cell in celltype_rename.values():
            df = evalute_fn(true_pkls[cell], pred_pkls[cell])
            df['celltype'] = cell
            df['feat_range'] = feat_range

            perform.append(df)
    except:
        print(cell)

featv5_df = pd.concat(perform, axis=0).reset_index().iloc[:, 1:]

In [8]:
featv5_df.sort_values(by=['Top5 events recall'],ascending=False)

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,feat_range
7,1.187942,3.363636,6.779347,0.801778,3.455428,7.010591,0.145256,3.695499,7.837599,0.055771,0.785395,0.768198,CHO,C200
4,1.111486,3.353045,6.586937,0.847792,3.403354,6.707855,0.146462,3.669020,7.611650,0.053288,0.850626,0.769500,K562,C100
2,1.333478,3.349515,6.789056,0.812693,3.469550,6.994704,0.154506,3.692851,7.809356,0.064535,0.784878,0.770782,CHO,C100
12,1.253783,3.336275,6.758164,0.793020,3.455428,7.012357,0.149746,3.703442,7.821712,0.057509,0.766625,0.766093,CHO,C500
9,1.145962,3.318623,6.574581,0.845628,3.375110,6.695499,0.151277,3.659312,7.606355,0.053564,0.852975,0.764080,K562,C200
14,1.218430,3.308032,6.553398,0.839652,3.367167,6.661959,0.155307,3.645190,7.590468,0.056116,0.847392,0.762157,K562,C500
3,1.185695,3.303619,6.383054,0.855528,3.411297,6.545455,0.149793,3.609885,7.520741,0.054744,0.830563,0.766862,HAP1,C100
6,1.401465,3.299206,6.501324,0.857485,3.418358,6.689320,0.197500,3.646072,7.926743,0.068274,0.768582,0.745756,mESC,C200
8,1.192699,3.296558,6.390997,0.848383,3.393645,6.545455,0.156893,3.599294,7.507502,0.056026,0.837235,0.762989,HAP1,C200
13,1.212467,3.295675,6.407767,0.848062,3.404237,6.557811,0.155753,3.616064,7.529568,0.056745,0.830224,0.762665,HAP1,C500


In [12]:
featv5_df.sort_values(by=['R2 of Frameshift ratio'],ascending=False)

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,feat_range
1,1.318381,3.283319,6.500441,0.861452,3.414828,6.650485,0.196301,3.646072,7.887908,0.065417,0.769938,0.743504,mESC,C100
6,1.401465,3.299206,6.501324,0.857485,3.418358,6.689320,0.197500,3.646072,7.926743,0.068274,0.768582,0.745756,mESC,C200
11,1.427277,3.261253,6.499559,0.857484,3.388350,6.667255,0.200569,3.627538,7.901147,0.067685,0.768625,0.744042,mESC,C500
3,1.185695,3.303619,6.383054,0.855528,3.411297,6.545455,0.149793,3.609885,7.520741,0.054744,0.830563,0.766862,HAP1,C100
0,2.014625,3.120035,5.831421,0.850139,3.225066,5.964695,0.237909,3.467785,7.029126,0.071106,0.790562,0.674679,iPSC,C100
5,2.206429,3.089144,5.808473,0.850075,3.207414,5.951456,0.240580,3.449250,7.023831,0.073393,0.787876,0.675714,iPSC,C200
10,2.167905,3.048544,5.757282,0.849797,3.178288,5.889673,0.247055,3.413063,7.001765,0.073362,0.782529,0.672925,iPSC,C500
8,1.192699,3.296558,6.390997,0.848383,3.393645,6.545455,0.156893,3.599294,7.507502,0.056026,0.837235,0.762989,HAP1,C200
13,1.212467,3.295675,6.407767,0.848062,3.404237,6.557811,0.155753,3.616064,7.529568,0.056745,0.830224,0.762665,HAP1,C500
4,1.111486,3.353045,6.586937,0.847792,3.403354,6.707855,0.146462,3.669020,7.611650,0.053288,0.850626,0.769500,K562,C100


In [13]:
featv5_df.query("`celltype` == 'iPSC'")

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,feat_range
0,2.014625,3.120035,5.831421,0.850139,3.225066,5.964695,0.237909,3.467785,7.029126,0.071106,0.790562,0.674679,iPSC,C100
5,2.206429,3.089144,5.808473,0.850075,3.207414,5.951456,0.240580,3.449250,7.023831,0.073393,0.787876,0.675714,iPSC,C200
10,2.167905,3.048544,5.757282,0.849797,3.178288,5.889673,0.247055,3.413063,7.001765,0.073362,0.782529,0.672925,iPSC,C500


In [56]:
featv5_df.query("`celltype` == 'mESC'")

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,feat_range
1,12.589545,0.000000,0.000883,0.003376,0.000000,0.000883,5.559279,0.016770,0.401589,0.260314,1.018913e-02,-0.019862,mESC,0:30
3,10.497226,0.017652,0.044131,0.117771,0.431598,1.058252,2.714250,0.069726,0.346867,0.254149,8.700070e-02,-0.014695,mESC,30:60
5,7.614685,1.567520,3.950574,0.233114,1.769638,4.385702,1.582771,0.483672,4.699029,0.238343,1.329089e-02,0.516663,mESC,60:90
7,11.417185,0.011474,0.032657,0.063203,0.011474,0.032657,3.483491,0.050309,0.388350,0.257159,6.346772e-02,-0.017030,mESC,90:120
9,9.951731,0.063548,0.388350,0.147419,0.972639,1.485437,2.377663,0.012357,0.223301,0.251458,2.294092e-03,-0.041050,mESC,120:150
11,9.396520,0.375993,1.589585,0.117081,0.947926,2.511915,2.067796,0.209179,1.522507,0.248059,3.137269e-02,0.257812,mESC,150:180
13,11.585173,0.002648,0.003530,0.011774,0.002648,0.003530,4.204984,0.012357,0.346867,0.258744,2.342687e-07,-0.023026,mESC,180:210
15,11.657201,0.015004,0.037952,0.064862,0.015004,0.038835,3.701280,0.068844,0.401589,0.259229,1.332920e-02,0.001881,mESC,210:240
17,11.518294,0.000000,0.002648,0.028381,0.000000,0.003530,3.603479,0.014122,0.346867,0.257579,3.383553e-02,-0.026672,mESC,240:270
19,7.523951,0.987643,3.370697,0.140685,1.465137,3.915269,1.525576,0.750221,5.353045,0.236603,1.022052e-01,0.570477,mESC,270:300


# read feature names

In [59]:
from inDecay import alignmap 

readFeaturesData = alignmap.fft.readFeaturesData

In [38]:
filename = "Oligo50293_GGATCGAGGCTATAAATCGC_features.txt"
feature_path = os.path.join(PATH.data_dir, 'IndelFeature_result', filename)

In [39]:
feature_df, cols = alignmap.fft.readFeaturesData(feature_path)
sinlge_col = [col for col in feature_df.columns if not col.startswith('PW_')]

feat_df = feature_df[sinlge_col]

In [42]:
feat_names = list(feat_df.columns)

In [44]:
for i in range(0,len(feat_names), 10):
    print(feat_names[i:i+10])

['Any Insertion', 'I1', 'I2', 'Any Deletion', 'D1', 'D2-3', 'D4-7', 'D8-12', 'D>12', 'DL-1--1']
['DL-2--2', 'DL-3--3', 'DL-4--6', 'DL-7--10', 'DL-11--15', 'DL-16--30', 'DL<-30', 'DL>=0', 'DR0-0', 'DR1-1']
['DR2-2', 'DR3-5', 'DR6-9', 'DR10-14', 'DR15-29', 'DR<0', 'DR=>30', 'IL-1--1', 'IL-2--2', 'IL-3--3']
['IL<-3', 'IL>=0', 'I1Rpt', 'I1NonRpt', 'I2Rpt', 'I2NonRpt', 'I1_A', 'I2_AA', 'I2_AT', 'I2_AG']
['I2_AC', 'I1_T', 'I2_TA', 'I2_TT', 'I2_TG', 'I2_TC', 'I1_G', 'I2_GA', 'I2_GT', 'I2_GG']
['I2_GC', 'I1_C', 'I2_CA', 'I2_CT', 'I2_CG', 'I2_CC', 'CS-5_NT=A', 'CS-5_NT=T', 'CS-5_NT=G', 'CS-5_NT=C']
['CS-4_NT=A', 'CS-4_NT=T', 'CS-4_NT=G', 'CS-4_NT=C', 'CS-3_NT=A', 'CS-3_NT=T', 'CS-3_NT=G', 'CS-3_NT=C', 'CS-2_NT=A', 'CS-2_NT=T']
['CS-2_NT=G', 'CS-2_NT=C', 'CS-1_NT=A', 'CS-1_NT=T', 'CS-1_NT=G', 'CS-1_NT=C', 'CS0_NT=A', 'CS0_NT=T', 'CS0_NT=G', 'CS0_NT=C']
['CS1_NT=A', 'CS1_NT=T', 'CS1_NT=G', 'CS1_NT=C', 'CS2_NT=A', 'CS2_NT=T', 'CS2_NT=G', 'CS2_NT=C', 'CS3_NT=A', 'CS3_NT=T']
['CS3_NT=G', 'CS3_NT=C',

In [49]:
for i in range(50, 120, 10):
    print(feat_names[i:i+10])

['I2_GC', 'I1_C', 'I2_CA', 'I2_CT', 'I2_CG', 'I2_CC', 'CS-5_NT=A', 'CS-5_NT=T', 'CS-5_NT=G', 'CS-5_NT=C']
['CS-4_NT=A', 'CS-4_NT=T', 'CS-4_NT=G', 'CS-4_NT=C', 'CS-3_NT=A', 'CS-3_NT=T', 'CS-3_NT=G', 'CS-3_NT=C', 'CS-2_NT=A', 'CS-2_NT=T']
['CS-2_NT=G', 'CS-2_NT=C', 'CS-1_NT=A', 'CS-1_NT=T', 'CS-1_NT=G', 'CS-1_NT=C', 'CS0_NT=A', 'CS0_NT=T', 'CS0_NT=G', 'CS0_NT=C']
['CS1_NT=A', 'CS1_NT=T', 'CS1_NT=G', 'CS1_NT=C', 'CS2_NT=A', 'CS2_NT=T', 'CS2_NT=G', 'CS2_NT=C', 'CS3_NT=A', 'CS3_NT=T']
['CS3_NT=G', 'CS3_NT=C', 'M_CS-2_-3_NT=A', 'M_CS-2_-3_NT=T', 'M_CS-2_-3_NT=G', 'M_CS-2_-3_NT=C', 'M_CS-1_-3_NT=A', 'M_CS-1_-3_NT=T', 'M_CS-1_-3_NT=G', 'M_CS-1_-3_NT=C']
['M_CS-1_-2_NT=A', 'M_CS-1_-2_NT=T', 'M_CS-1_-2_NT=G', 'M_CS-1_-2_NT=C', 'M_CS0_-3_NT=A', 'M_CS0_-3_NT=T', 'M_CS0_-3_NT=G', 'M_CS0_-3_NT=C', 'M_CS0_-2_NT=A', 'M_CS0_-2_NT=T']
['M_CS0_-2_NT=G', 'M_CS0_-2_NT=C', 'M_CS0_-1_NT=A', 'M_CS0_-1_NT=T', 'M_CS0_-1_NT=G', 'M_CS0_-1_NT=C', 'M_CS1_-3_NT=A', 'M_CS1_-3_NT=T', 'M_CS1_-3_NT=G', 'M_CS1_-3_NT=C']


# read_data

In [60]:
from scripts.STfeatv5_inDecay import read_data

In [57]:
from inDecay import reader
from inDecay import ForeCast_features as fft

In [61]:
reference_path = os.path.join(PATH.data_dir, "SelfTarget_NewScaffold.fasta")
ref_lookup = reader.get_reference()

In [66]:
test_Oligo = "Oligo50293"
Guide, refseq, pamsite, Strand = ref_lookup[test_Oligo]
idfgen_file = my_utils.get_indelgen_file(test_Oligo, Guide)
idfgen = pd.read_table(idfgen_file, skiprows=1, names=['Identifier', 'n_coevent', 'loc'])

In [67]:
idfgen

,Identifier,n_coevent,loc
0,D10_L-10C2R3,3,"[(43,54,),(44,55,),(45,56,)]"
1,D10_L-11R0,1,"[(42,53,)]"
2,D10_L-1R10,1,"[(52,63,)]"
3,D10_L-2R9,1,"[(51,62,)]"
4,D10_L-3R8,1,"[(50,61,)]"
...,...,...,...
374,I2_L-1C4R4,1,"[(52,53,CG)]"
375,I2_L-1R0,9,"[(52,53,AA),(52,53,TA),(52,53,GA),(52,53,AG),(..."
376,I2_L-2C1R0,2,"[(52,53,TT),(52,53,GT)]"
377,I2_L-2C2R1,1,"[(52,53,CT)]"


In [71]:
import Bio

In [117]:
A,T,G,C = 'A','T','G','C'
AA,AT,AC,AG,CG,CT,CA,CC = 'AA','AT','AC','AG','CG','CT','CA','CC'
GT,GA,GG,GC,TA,TG,TC,TT = 'GT','GA','GG','GC','TA','TG','TC','TT'

is_reverse = (Strand == "REVERSE")

cut_site = pamsite - 3

feature_matrix = []
indels = []
for i, row in idfgen.iterrows():
    indel = row['Identifier']
    indel_locs = eval(row['loc'])
    for indel_loc in indel_locs:
        ins_seq = indel_loc[2] if len(indel_loc) > 2 else ''
        left = indel_loc[0] if not is_reverse else (78 - indel_loc[1]) 
        right = indel_loc[1] if not is_reverse else (78 - indel_loc[0]) 
        ins_seq = ins_seq if not is_reverse else Bio.Seq.reverse_complement(ins_seq)
        indel_details = (refseq, cut_site, left, right, ins_seq)

        lin_fts, feat_name = fft.features_SeqMatches(indel_details)

        feature_matrix.append(lin_fts)
        indels.append(indel)

feature_df = pd.DataFrame(feature_matrix, columns=feat_name)
feature_df['Identifier'] = indels

In [120]:
feature_df.groupby('Identifier').any()

,M_L-3_R-3,X_L-3_R-3,M_L-3_R-2,X_L-3_R-2,M_L-3_R-1,X_L-3_R-1,M_L-3_R0,X_L-3_R0,M_L-3_R1,X_L-3_R1,...,M_L2_R-2,X_L2_R-2,M_L2_R-1,X_L2_R-1,M_L2_R0,X_L2_R0,M_L2_R1,X_L2_R1,M_L2_R2,X_L2_R2
Identifier,,,,,,,,,,,,,,,,,,,,,
D10_L-10C2R3,False,True,True,True,False,True,True,True,True,True,...,True,True,False,True,False,True,False,True,False,True
D10_L-11R0,False,True,False,True,False,True,True,False,False,True,...,False,True,False,True,True,False,False,True,True,False
D10_L-1R10,False,True,False,True,False,True,False,True,True,False,...,True,False,False,True,False,True,False,True,False,True
D10_L-2R9,False,True,False,True,False,True,False,True,False,True,...,False,True,False,True,True,False,True,False,False,True
D10_L-3R8,False,True,True,False,False,True,False,True,False,True,...,False,True,True,False,True,False,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
I2_L-1C4R4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
I2_L-1R0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
I2_L-2C1R0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [116]:
feature_matrix = []
for x in cal_feat_input:
    all_features = []
    lin_fts, feat_name = fft.features_SeqMatches(x)
    # for fname in lin_fts:
    #     all_features.extend(lin_fts[fname][0])

    feature_matrix.append(lin_fts)
    
feature_matrix = np.stack(feature_matrix)
feature_df = pd.DataFrame(feature_matrix, columns=feat_name)

In [102]:
feature_matrix.shape

(500, 40)

In [108]:
func_2_feat = {k:list(v)[1] for k,v in lin_fts[0].items()}

AttributeError: 'bool' object has no attribute 'items'

In [114]:
feature_list = ["feature_InsSize", "feature_DelSize", "feature_DelLoc", "feature_InsLoc", "feature_I1or2Rpt","feature_InsSeq", "feature_LocalCutSiteSequence", "feature_LocalCutSiteSeqMatches", "feature_LocalRelativeSequence", "features_SeqMatches", "feature_microhomology"]

cumsum_i = 0
for feat in feature_list:
    len_feat = len(func_2_feat[feat][1])
    print(feat, cumsum_i, ":", cumsum_i + len_feat)
    cumsum_i += len_feat

feature_InsSize 0 : 3
feature_DelSize 3 : 9
feature_DelLoc 9 : 27
feature_InsLoc 27 : 32
feature_I1or2Rpt 32 : 36
feature_InsSeq 36 : 56
feature_LocalCutSiteSequence 56 : 92
feature_LocalCutSiteSeqMatches 92 : 132
feature_LocalRelativeSequence 132 : 180
features_SeqMatches 180 : 252
feature_microhomology 252 : 273


In [115]:
92 - 56

36

In [89]:
list(lin_fts[0].values())[0][0]

([False, False, False], ['Any Insertion', 'I1', 'I2'])

In [84]:
feature_X = [list(fts_dict.values())[0] for fts_dict in lin_fts]

In [85]:
feature_X

[([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any Insertion', 'I1', 'I2']),
 ([False, False, False], ['Any 